<a href="https://colab.research.google.com/github/wuqichuang/learning-python/blob/main/notebooks/07_day16-20_advanced.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 在浏览器中打开本 notebook 后，点击菜单 **代码执行程序 → 全部运行** 即可按顺序执行所有单元格。

# Day 16–20｜进阶五天：SQL 进阶 · 正则清洗 · 窗口函数 · 网页化 · 备份

项目已经完成，这 5 天把每件武器都再磨快一档——这些内容正是**科研数据处理**中最常用的进阶技能。

| 天 | 主题 | 为什么值得学 |
|---|---|---|
| Day 16 | SQL 进阶：子查询与视图 | 一次查询搞定"比平均分高的人"这类复杂问题 |
| Day 17 | 正则表达式批量清洗文本 | 处理文献、日志、实验记录里脏数据的利器 |
| Day 18 | 窗口函数 | 排名、累计、组内对比——科研统计分析高频 |
| Day 19 | Flask 网页演示 + 数据备份 | 把系统给别人用；让数据永不丢失 |
| Day 20 | 总复盘 + 结营测验 | 把 20 天织成一张完整的知识网 |

**节奏**：每天 4 小时 = 讲解 1 小时 + 练习 2.5 小时 + 答疑 0.5 小时。

> 本 notebook 中 SQL 演示用 sqlite（Colab 可运行）；同样的语句在你的 MySQL 里基本都能直接跑（个别差异已标注）。

---
## 🗓️ Day 16｜SQL 进阶：子查询与视图

### 子查询：查询里套查询

"找出数学成绩高于全班数学平均分的人"——需要两步：先算平均分，再比较。子查询把两步合成一句。

In [ ]:
import sqlite3, pandas as pd

# 准备演示数据（沿用学生管理系统的三张表，重新造一份）
conn = sqlite3.connect("adv.db")
conn.executescript("""
DROP TABLE IF EXISTS scores; DROP TABLE IF EXISTS students; DROP TABLE IF EXISTS classes;
CREATE TABLE classes (id INTEGER PRIMARY KEY AUTOINCREMENT, cls_name TEXT UNIQUE, teacher TEXT);
CREATE TABLE students (id INTEGER PRIMARY KEY AUTOINCREMENT, name TEXT, cls_id INT, gender TEXT, age INT);
CREATE TABLE scores (id INTEGER PRIMARY KEY AUTOINCREMENT, stu_id INT, subject TEXT, score REAL, UNIQUE(stu_id, subject));
INSERT INTO classes (cls_name, teacher) VALUES ('一班','张老师'),('二班','王老师');
INSERT INTO students VALUES (1,'李雷',1,'男',17),(2,'韩梅梅',2,'女',17),(3,'Jim',1,'男',18),
                            (4,'Lucy',2,'女',17),(5,'Lily',1,'女',18),(6,'王强',2,'男',17);
INSERT INTO scores (stu_id, subject, score) VALUES
(1,'数学',88),(1,'英语',92),(1,'语文',85),(2,'数学',91),(2,'英语',84),(3,'数学',76),(3,'英语',88),
(3,'语文',90),(4,'数学',59),(4,'英语',71),(5,'数学',95),(5,'英语',60),(5,'语文',78),(6,'数学',83),(6,'英语',79);
""")
conn.commit()

# 子查询：高于全班数学平均分的学生
print(pd.read_sql("""
    SELECT s.name, sc.score FROM scores sc JOIN students s ON sc.stu_id = s.id
    WHERE sc.subject = '数学'
      AND sc.score > (SELECT AVG(score) FROM scores WHERE subject = '数学')
""", conn))

In [ ]:
# IN 子查询：找出"没有任何一科及格"的学生（演示 NOT IN）
r = pd.read_sql("""
    SELECT name FROM students
    WHERE id NOT IN (SELECT DISTINCT stu_id FROM scores WHERE score >= 60)
""", conn)
print(r if len(r) else "（全部都有及格科目）")

# FROM 里的子查询：先汇总再统计——"各班总平均分"
print(pd.read_sql("""
    SELECT cls_name, ROUND(AVG(total),1) AS 班级总平均 FROM (
        SELECT s.id, c.cls_name, SUM(sc.score) AS total
        FROM scores sc
        JOIN students s ON sc.stu_id = s.id
        JOIN classes  c ON s.cls_id = c.id
        GROUP BY s.id
    ) t GROUP BY cls_name
""", conn))

### 视图 VIEW：把复杂查询存成"虚拟表"

写好的复杂 JOIN 查询可以存成视图，以后当表用——MySQL 和 sqlite 都支持。

```sql
CREATE VIEW v_student_total AS
SELECT s.id, s.name, c.cls_name, SUM(sc.score) AS total, AVG(sc.score) AS avg_score
FROM scores sc
JOIN students s ON sc.stu_id = s.id
JOIN classes  c ON s.cls_id = c.id
GROUP BY s.id;

SELECT * FROM v_student_total ORDER BY total DESC;   -- 直接当表查
```

In [ ]:
conn.execute("DROP VIEW IF EXISTS v_student_total")
conn.execute("""
    CREATE VIEW v_student_total AS
    SELECT s.id, s.name, c.cls_name, SUM(sc.score) AS total, ROUND(AVG(sc.score),1) AS avg_score
    FROM scores sc
    JOIN students s ON sc.stu_id = s.id
    JOIN classes  c ON s.cls_id = c.id
    GROUP BY s.id
""")
print(pd.read_sql("SELECT * FROM v_student_total ORDER BY total DESC", conn))

### ✏️ Day 16 练习清单
1. 用子查询找出英语成绩**低于**全班英语平均分的学生
2. 用子查询找出"总分最高的学生姓名和总分"（提示：总分用 FROM 子查询先算）
3. 创建一个视图 `v_subject_stat`：每科目的平均分、最高分、最低分、及格率，并查询它
4. （挑战，在服务器 MySQL 上做）把项目数据库里也建一个 `v_student_total` 视图

---
## 🗓️ Day 17｜正则表达式：批量清洗文本

### re 模块四个核心函数

| 函数 | 作用 |
|---|---|
| `re.findall(pattern, text)` | 找出所有匹配，返回列表 |
| `re.search(pattern, text)` | 找第一个匹配（可分组提取） |
| `re.sub(pattern, repl, text)` | 替换所有匹配 |
| `re.split(pattern, text)` | 按匹配拆分 |

### 常用符号速查

`\d` 数字 · `\w` 字母数字下划线 · `\s` 空白 · `.` 任意字符 · `*` 0 次或多次 · `+` 1 次或多次 · `{n,m}` n 到 m 次 · `[abc]` 其中之一 · `()` 分组提取

In [ ]:
import re

# 场景 1：从实验日志里批量提取数字
log = "样本A: 23.5mg, 样本B: 41.2mg, 样本C: 38.0mg"
values = re.findall(r"\d+\.\d+", log)
print("提取的数值：", values)

# 场景 2：分组提取——从混乱记录里提取学号和成绩
records = "学号2026001成绩88; 学号2026002成绩91; 学号2026003成绩76"
pairs = re.findall(r"学号(\d+)成绩(\d+)", records)
print("学号-成绩对：", pairs)

# 场景 3：统一日期格式 2026/07/17 → 2026-07-17
text = "实验日期 2026/07/17，复查日期 2026/08/02"
print(re.sub(r"(\d{4})/(\d{2})/(\d{2})", r"\1-\2-\3", text))

# 场景 4：清洗多余空白
messy = "李雷 ,  韩梅梅,Jim   ,Lucy"
names = [x.strip() for x in re.split(r"[,，]", messy)]
print(names)

In [ ]:
# 场景 5（科研高频）：校验数据格式是否合法
def is_valid_sid(s):
    """学号必须是 2026 开头的 8 位数字"""
    return bool(re.fullmatch(r"2026\d{4}", s))

def is_valid_email(s):
    return bool(re.fullmatch(r"[\w.]+@[\w]+\.[a-z]{2,}", s))

for x in ["20260001", "2026abc1", "202600"]:
    print(x, is_valid_sid(x))
for x in ["zhang.san@example.com", "not-an-email"]:
    print(x, is_valid_email(x))

### ✏️ Day 17 练习清单
1. 从 `"联系: 138-1234-5678 或 139-8765-4321"` 提取所有手机号（`\d{3}-\d{4}-\d{4}`）
2. 把 `"2026年07月17日"` 改写成 `2026-07-17`（用分组替换）
3. 写函数 `extract_scores(text)`：从 `"数学:88, 英语:92, 语文:85"` 中提取成字典 `{"数学": 88, ...}`
4. （挑战）从一段英文文献摘要里提取所有年份（19xx 或 20xx）

---
## 🗓️ Day 18｜窗口函数：排名与组内对比

普通 `GROUP BY` 会把多行压成一行；**窗口函数 `OVER()` 保留每一行**，同时给它挂上"组内统计值"——排名、占比、累计、与平均的差距。

> sqlite 和 MySQL 8.0+ 都支持窗口函数，语法一致。

In [ ]:
# 1) RANK()：全班总分排名（每行都保留明细）
print(pd.read_sql("""
    SELECT name, total,
           RANK() OVER (ORDER BY total DESC) AS 排名
    FROM v_student_total
""", conn))

# 2) PARTITION BY：组内排名——每班内部的数学成绩排名
print(pd.read_sql("""
    SELECT c.cls_name AS 班级, s.name, sc.score,
           RANK() OVER (PARTITION BY c.cls_name ORDER BY sc.score DESC) AS 班内排名
    FROM scores sc
    JOIN students s ON sc.stu_id = s.id
    JOIN classes  c ON s.cls_id  = c.id
    WHERE sc.subject = '数学'
""", conn))

In [ ]:
# 3) 每行挂组内统计：每个学生的总分与班级平均的差距
print(pd.read_sql("""
    SELECT name, cls_name, total,
           ROUND(AVG(total) OVER (PARTITION BY cls_name),1) AS 班级平均,
           ROUND(total - AVG(total) OVER (PARTITION BY cls_name),1) AS 与班平均差距
    FROM v_student_total
""", conn))

# 4) 累计值：按总分从高到低算累计人数占比（科研里画累积分布常用）
print(pd.read_sql("""
    SELECT name, total,
           ROUND(100.0 * ROW_NUMBER() OVER (ORDER BY total DESC) / COUNT(*) OVER (), 1) AS '累计占比%'
    FROM v_student_total
""", conn))

### ✏️ Day 18 练习清单
1. 用窗口函数给出**每班英语成绩第 1 名**的学生（提示：PARTITION BY + RANK，外层筛 排名=1）
2. 给 v_student_total 加一列"年级平均"（`AVG(total) OVER ()` 不带 PARTITION）
3. 计算每个学生的总分占**全班总分**的百分比
4. （挑战）用 `LAG(total) OVER (ORDER BY total DESC)` 算出每人和前一名的分差

---
## 🗓️ Day 19｜Flask 网页演示 + 数据备份

### 上午：把学生管理系统"长"出一个网页界面

下面是一个最小 Flask 应用：浏览器访问就能看到学生列表、可以添加学生。**在云服务器上运行**，本地浏览器通过 `http://服务器IP:5000` 访问。

In [ ]:
# 本单元格会把网页程序写成 web_app.py 文件。
# 部署：scp 上传到服务器 → pip3 install flask → nohup python3 web_app.py > web.log 2>&1 &
# （安全组需放行 5000 端口；学习用途，了解即可）

web_app = '''
import sqlite3
from flask import Flask, request, redirect

app = Flask(__name__)
DB = "student_mgmt.db"   # 服务器上换成 pymysql 连接你的 MySQL

PAGE = """<h1>学生管理系统</h1>
<form method="post" action="/add">
  姓名 <input name="name"> 班级 <input name="cls">
  性别 <input name="gender"> 年龄 <input name="age" size="4">
  <button type="submit">添加</button>
</form>
<table border="1" cellpadding="6">
<tr><th>学号</th><th>姓名</th><th>班级</th><th>性别</th><th>年龄</th></tr>
{rows}
</table>"""


def query_students():
    conn = sqlite3.connect(DB)
    rows = conn.execute(
        "SELECT s.id, s.name, c.cls_name, s.gender, s.age "
        "FROM students s JOIN classes c ON s.cls_id=c.id"
    ).fetchall()
    conn.close()
    return rows


@app.route("/")
def index():
    trs = "".join(
        f"<tr><td>{r[0]}</td><td>{r[1]}</td><td>{r[2]}</td>"
        f"<td>{r[3]}</td><td>{r[4]}</td></tr>"
        for r in query_students()
    )
    return PAGE.format(rows=trs)


@app.route("/add", methods=["POST"])
def add():
    f = request.form
    conn = sqlite3.connect(DB)
    conn.execute("INSERT OR IGNORE INTO classes (cls_name) VALUES (?)", (f["cls"],))
    cls_id = conn.execute("SELECT id FROM classes WHERE cls_name=?", (f["cls"],)).fetchone()[0]
    conn.execute("INSERT INTO students (name, cls_id, gender, age) VALUES (?,?,?,?)",
                 (f["name"], cls_id, f["gender"], int(f["age"])))
    conn.commit()
    conn.close()
    return redirect("/")


if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)
'''

with open("web_app.py", "w", encoding="utf-8") as f:
    f.write(web_app)
print("✅ 已生成 web_app.py，共", len(web_app), "字符")
import ast
ast.parse(open("web_app.py", encoding="utf-8").read())
print("✅ web_app.py 语法检查通过")

### 下午：数据备份——给系统上保险

**mysqldump 备份**（在服务器终端执行）：

```bash
# 手动备份一次
mkdir -p ~/backups
mysqldump -u learn -p student_mgmt > ~/backups/student_$(date +%Y%m%d).sql
ls -lh ~/backups/

# 恢复（演示用，别对正式库乱试）
# mysql -u learn -p student_mgmt < ~/backups/student_20260719.sql
```

**crontab 定时自动备份**（每天凌晨 3 点自动执行）：

```bash
crontab -e        # 编辑定时任务，加入下面这行：
0 3 * * * mysqldump -u learn -p'Learn@2026' student_mgmt > ~/backups/student_$(date +\%Y\%m\%d).sql
```

> 💡 `crontab -l` 查看已有任务；`date` 命令的 `%` 在 crontab 里要写成 `\%`。

### ✏️ Day 19 练习清单
1. 在服务器上跑通 web_app.py，浏览器访问看到学生列表，并成功添加 1 名学生
2. 手动执行一次 mysqldump 备份，用 `head` 看看备份文件内容
3. 配置 crontab 每日备份，用 `crontab -l` 确认；第二天检查 backups 目录里有没有新文件
4. （挑战）给网页再加一个"按班级筛选"的下拉框

---
## 🗓️ Day 20｜总复盘 + 结营

### 上午：画出自己的知识地图（不看资料，凭记忆写，写完再核对）

1. **Python/字符串**：切片怎么取？f-string 三种用法？split 和 join 互逆吗？
2. **Pandas**：筛行、选列、新增列、groupby、merge、read_csv 各怎么写？
3. **Linux**：ssh、cd/ls、cp/mv/rm、chmod、apt、ps/kill、nohup、crontab 各干什么？
4. **MySQL**：建表、插入、查询（WHERE/ORDER BY/LIMIT）、UPDATE/DELETE、GROUP BY/HAVING、JOIN、子查询、视图、窗口函数
5. **项目**：你的学生管理系统数据怎么流的？（CSV/输入 → Python → SQL → MySQL → 查询 → 报表/网页）

### 下午：结营三件事
1. 做【测验 4】（进阶综合）
2. 把 20 天的问题清单整理成一份"我的 FAQ"文档（以后复习用）
3. 给自己写一个"下一步计划"：博士申请中打算用这套技能做什么（比如：文献数据整理、实验数据自动统计）

---
## 🎉 20 天结业

到这里，你已经完成了：**Python 字符串与数据处理 → Pandas → Linux 服务器实操 → MySQL 数据库 → 完整的学生管理系统（含部署、网页、备份）→ SQL/文本/统计进阶**。

这套"Python + 数据库 + 服务器"的组合，正是科研数据处理的核心工具链。继续保持每天动手的习惯，祝博士申请顺利！

---
## 📒 我的问题清单（随手记录，答疑前整理）

学习中遇到任何卡住的地方，立刻记在下面表格里，**答疑时间集中解决**：

| 日期 | 问题描述 | 状态（待问/已解决） | 答案要点 |
|---|---|---|---|
|  |  |  |  |
|  |  |  |  |
|  |  |  |  |
|  |  |  |  |

---
## ✅ 参考答案（先自己做，再对照）

```sql
-- Day16-1
SELECT s.name, sc.score FROM scores sc JOIN students s ON sc.stu_id=s.id
WHERE sc.subject='英语' AND sc.score < (SELECT AVG(score) FROM scores WHERE subject='英语');
-- Day16-2
SELECT name, total FROM (SELECT s.id, s.name, SUM(sc.score) total
  FROM scores sc JOIN students s ON sc.stu_id=s.id GROUP BY s.id) t
ORDER BY total DESC LIMIT 1;

-- Day18-1
SELECT 班级, name, score FROM (
  SELECT c.cls_name AS 班级, s.name, sc.score,
         RANK() OVER (PARTITION BY c.cls_name ORDER BY sc.score DESC) AS rk
  FROM scores sc JOIN students s ON sc.stu_id=s.id JOIN classes c ON s.cls_id=c.id
  WHERE sc.subject='英语') t WHERE rk = 1;
-- Day18-3
SELECT name, total, ROUND(100.0*total/SUM(total) OVER (),1) AS '占比%' FROM v_student_total;
```

```python
# Day17-1
import re
re.findall(r"\d{3}-\d{4}-\d{4}", "联系: 138-1234-5678 或 139-8765-4321")
# Day17-2
re.sub(r"(\d{4})年(\d{2})月(\d{2})日", r"\1-\2-\3", "2026年07月17日")
# Day17-3
def extract_scores(text):
    return {k: int(v) for k, v in re.findall(r"(\w+):(\d+)", text)}
```